# Geometric EEG SSL — Experiment Notebook (GPU)

**Purpose:** Run the full eval suite (E1, E2 a/b/c, E5, E7) against the
pretrained checkpoints. Short GPU work compared to pretraining; safe to
re-run anytime as you iterate on probe code.

**Prereqs:**
- All three datasets cached via `colab_download.ipynb`.
- At least G1 + Codex checkpoints from `colab_pretrain.ipynb`
  (Channel-Independent is required for E1/E7; G2/G3 for E5).

**Before running:** Runtime → Change runtime type → GPU.

## 0. GPU check

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: no GPU detected — set Runtime → Change runtime type → GPU')
    sys.exit(1)

## 1. Install dependencies

In [ ]:
%%capture
!pip install mne moabb scikit-learn pyyaml scipy

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/geometric_eeg_ssl'
os.makedirs(DRIVE_ROOT, exist_ok=True)

# MNE data cached here — avoids re-downloading across sessions
MNE_DATA_DIR = f'{DRIVE_ROOT}/mne_data'
os.makedirs(MNE_DATA_DIR, exist_ok=True)
os.environ['MNE_DATA'] = MNE_DATA_DIR

# Checkpoints saved here
CKPT_ROOT = f'{DRIVE_ROOT}/runs'
os.makedirs(CKPT_ROOT, exist_ok=True)
print(f'Drive mounted. Checkpoints → {CKPT_ROOT}')
# Read preprocessing cache built by colab_download.ipynb.
CACHE_ROOT = f'{DRIVE_ROOT}/cache'
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT
if not os.path.isdir(CACHE_ROOT):
    print(f'WARNING: no cache at {CACHE_ROOT}. Eval will recompute preprocessing.')
else:
    print(f'Preprocessing cache → {CACHE_ROOT}')


## 3. Clone repo

In [ ]:
import os, sys
REPO_DIR = '/content/geometric-eeg-ssl'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tianxin-scu/geometric-eeg-ssl.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, f'{REPO_DIR}/src')
print('Repo ready at', REPO_DIR)

## 4. Wire repo to Drive checkpoints

Symlinks `{REPO_DIR}/runs` → `{CKPT_ROOT}` so the eval scripts find the
`*_full/epoch_*.pt` checkpoints via their default `runs/pretrain/` path.

In [ ]:
import os, subprocess, sys

os.environ['MNE_DATA'] = MNE_DATA_DIR

# Symlink Drive runs/ into repo so eval scripts find checkpoints via their
# default REPO_ROOT/runs/pretrain/{name}_full/ path.
RUNS_LINK = f'{REPO_DIR}/runs'
RUNS_TARGET = f'{CKPT_ROOT}'
if not os.path.exists(RUNS_LINK):
    os.symlink(RUNS_TARGET, RUNS_LINK)
    print(f'Symlinked {RUNS_LINK} -> {RUNS_TARGET}')
else:
    print(f'runs/ link already exists: {os.path.realpath(RUNS_LINK)}')

## 5. Checkpoint inventory (sanity check)

In [ ]:
import glob, os

VARIANTS = [
    ('G1',                 f'{CKPT_ROOT}/g1_full'),
    ('G2',                 f'{CKPT_ROOT}/g2_full'),
    ('G3',                 f'{CKPT_ROOT}/g3_full'),
    ('Transductive Codex', f'{CKPT_ROOT}/codex_full'),
    ('Channel-Indep',      f'{CKPT_ROOT}/chind_full'),
]

print(f"{'Variant':<25} {'Latest checkpoint'}")
print('-' * 60)
for label, ckpt_dir in VARIANTS:
    ckpts = sorted(glob.glob(f'{ckpt_dir}/epoch_*.pt'))
    if ckpts:
        latest = os.path.basename(ckpts[-1])
        n = len(ckpts)
        print(f'  {label:<23} {latest}  ({n} saved)')
    else:
        print(f'  {label:<23} (no checkpoints)')

## 6. E1 — In-Distribution Linear Probe

PhysioNet MI LOSO: G1 vs. Transductive Codex vs. Channel-Independent.
Fairness anchor for all robustness claims.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT

!python {REPO_DIR}/scripts/run_e1.py \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e1_log.txt

## 7. E2(a) — Sleep-EDFx cross-session (night 1 → night 2)

Probes G1 and Transductive Codex on Sleep-EDFx within-subject night split.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT

!python {REPO_DIR}/scripts/run_e2.py a \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e2a_log.txt

## 7b. E2(b) — PhysioNet MI LOSO + Wilcoxon test

LOSO BAC for G1 vs. Codex with paired Wilcoxon signed-rank test.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT

!python {REPO_DIR}/scripts/run_e2.py b \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e2b_log.txt

## 7c. E2(c) — Cross-montage zero-shot to BCIC-2B

64-ch PhysioNet MI pretrain → 3-ch BCIC-2B evaluation.
Transductive Codex is tested with both `random` and `nn` codex fallbacks.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT

!python {REPO_DIR}/scripts/run_e2.py c \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e2c_log.txt

## 8. E5 — G1 / G2 / G3 Ablation

Where in attention does g_ij enter? Reports BAC on PhysioNet MI (in-distribution)
and BCIC-2B (cross-montage), plus distinguishing parameter counts.

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT

!python {REPO_DIR}/scripts/run_e5.py \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e5_log.txt

## 9. E7 — Channel-Independent Sanity Check

Does explicit spatial structure help at all? Channel-Independent vs. G1 vs. Codex
on PhysioNet MI (in-distribution) and BCIC-2B (cross-montage).

In [ ]:
import os
os.environ['MNE_DATA'] = MNE_DATA_DIR
os.environ['EEG_CACHE_DIR'] = CACHE_ROOT

!python {REPO_DIR}/scripts/run_e7.py \
    --device cuda \
    --out-tag full \
    2>&1 | tee {CKPT_ROOT}/e7_log.txt

## 10. Full results summary

In [ ]:
import json, glob, os

# ── Checkpoint inventory ──────────────────────────────────────────────────
VARIANTS = [
    ('G1',                 f'{CKPT_ROOT}/g1_full'),
    ('G2',                 f'{CKPT_ROOT}/g2_full'),
    ('G3',                 f'{CKPT_ROOT}/g3_full'),
    ('Transductive Codex', f'{CKPT_ROOT}/codex_full'),
    ('Channel-Indep',      f'{CKPT_ROOT}/chind_full'),
]

print('=== Checkpoint inventory ===')
for label, ckpt_dir in VARIANTS:
    ckpts = sorted(glob.glob(f'{ckpt_dir}/epoch_*.pt'))
    status = os.path.basename(ckpts[-1]) if ckpts else '(missing)'
    print(f'  {label:<25} {status}')

# ── Eval log summary ─────────────────────────────────────────────────────
print()
print('=== Eval logs (last 5 lines each) ===')
for tag, logfile in [
    ('E1',  f'{CKPT_ROOT}/e1_log.txt'),
    ('E2a', f'{CKPT_ROOT}/e2a_log.txt'),
    ('E2b', f'{CKPT_ROOT}/e2b_log.txt'),
    ('E2c', f'{CKPT_ROOT}/e2c_log.txt'),
    ('E5',  f'{CKPT_ROOT}/e5_log.txt'),
    ('E7',  f'{CKPT_ROOT}/e7_log.txt'),
]:
    if os.path.exists(logfile):
        lines = open(logfile).readlines()
        print(f'\n--- {tag} ---')
        print(''.join(lines[-5:]).strip())
    else:
        print(f'\n--- {tag} --- (not run yet)')